In [ ]:
import matplotlib.pyplot as plt
import nocte.core.matching
import nocte.core.traces
import nocte.core.windows
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------
# Synthetic dataset
# ---------------------------------------------------------------------

rng = np.random.default_rng(12345)

hz = 100.0
duration_ms = 12_000.0
period_ms = 1000.0 / hz

time = np.arange(0.0, duration_ms, period_ms)

experiments = ['A', 'B', 'C']
channels = [0, 1]

trace_meta = pd.DataFrame(
    [
        {'experiment': experiment, 'channel': channel}
        for experiment in experiments
        for channel in channels
    ],
    index=pd.RangeIndex(6, name='trace'),
)


# Event/reference times are deliberately DIFFERENT between experiments.
#
# They are also deliberately not aligned to the native 10-ms sample grid.
# That makes resampling/alignment do some actual work.
event_times = {
    'A': [1803.0, 4207.0, 6811.0, 9303.0],
    'B': [2407.0, 5003.0, 7511.0, 10_107.0],
    'C': [1301.0, 3607.0, 6103.0, 8807.0],
}


# Slightly different strength on each repetition.
trial_gains = [0.85, 1.00, 1.15, 0.95]

# Experiment and channel amplitudes make it easy to spot provenance errors.
experiment_gain = {
    'A': 1.0,
    'B': 1.5,
    'C': 2.0,
}

channel_gain = {
    0: 1.0,
    1: 0.65,
}


def response(dt):
    """
    Fixed waveform added around every event.

    dt is milliseconds relative to the event reference.
    """
    positive = np.exp(-0.5 * ((dt - 80.0) / 55.0) ** 2)
    negative = 0.45 * np.exp(-0.5 * ((dt - 230.0) / 90.0) ** 2)

    return positive - negative


# Low-level independent noise.
values = rng.normal(
    loc=0.0,
    scale=0.05,
    size=(len(trace_meta), len(time)),
)


# Add responses ONLY at the event times belonging to each trace's experiment.
for trace_pos, (_, props) in enumerate(trace_meta.iterrows()):
    experiment = props['experiment']
    channel = props['channel']

    for event_time, trial_gain in zip(
        event_times[experiment],
        trial_gains,
        strict=True,
    ):
        values[trace_pos] += (
            experiment_gain[experiment]
            * channel_gain[channel]
            * trial_gain
            * response(time - event_time)
        )


traces = nocte.core.traces.Traces.from_array(
    values,
    hz=hz,
    start=0.0,
    meta=trace_meta,
)


# ---------------------------------------------------------------------
# Windows
# ---------------------------------------------------------------------

window_meta_rows = []
window_refs = []

for experiment in experiments:
    for trial, event_time in enumerate(event_times[experiment]):
        window_refs.append(event_time)

        window_meta_rows.append(
            {
                'experiment': experiment,
                'trial': trial,
                'condition': 'even' if trial % 2 == 0 else 'odd',
            }
        )

win_meta = pd.DataFrame(
    window_meta_rows,
    index=pd.RangeIndex(len(window_meta_rows), name='win'),
)

window_refs = np.asarray(window_refs)


# Asymmetric on purpose:
#
#     ref
#      |
#  ----|--------
# -500       +800 ms
#
# This makes align='ref' and align='mid' visibly different.
wins = nocte.core.windows.Windows.build_around(
    window_refs,
    nocte.core.windows.Win(-500.0, 800.0),
    meta=win_meta,
)


print('Traces:')
display(traces.meta)

print('Windows:')
display(
    wins.meta.assign(
        ref=wins.ref,
        start=wins.time_at('start'),
        stop=wins.time_at('stop'),
    )
)


# ---------------------------------------------------------------------
# Plot the original synthetic traces
# ---------------------------------------------------------------------

fig, axs = plt.subplots(
    3,
    2,
    figsize=(12, 7),
    sharex=True,
    constrained_layout=True,
)

for ax, (trace_id, trace) in zip(axs.flat, traces.items(), strict=True):
    props = traces.meta.loc[trace_id]
    experiment = props['experiment']
    channel = props['channel']

    ax.plot(trace.index, trace.values, linewidth=0.8)

    for event_time in event_times[experiment]:
        ax.axvline(event_time, linewidth=0.7, alpha=0.35)

    ax.set_title(f'trace {trace_id}: experiment {experiment}, channel {channel}')
    ax.set_ylabel('signal')

for ax in axs[-1]:
    ax.set_xlabel('time [ms]')

fig.suptitle('Synthetic source traces — vertical lines are the matching events')
plt.show()


# ---------------------------------------------------------------------
# Explicit matching
# ---------------------------------------------------------------------

matches = nocte.core.matching.Matches.from_meta(
    traces,
    wins,
    by='experiment',
)

print(f'{len(traces) = }')
print(f'{len(wins) = }')
print(f'{len(matches) = }')

# Expected:
#
# 3 experiments
# × 2 traces per experiment
# × 4 windows per experiment
# = 24 matches
assert len(matches) == 24


# Make the relation human-readable.
pairs = matches.to_frame()

pairs = pairs.join(
    traces.meta.add_prefix('trace_'),
    on='trace',
).join(
    wins.meta.add_prefix('win_'),
    on='win',
)

display(pairs)

# Every explicit match should pair the same experiment.
assert (pairs['trace_experiment'] == pairs['win_experiment']).all()


# ---------------------------------------------------------------------
# Exercise the four extraction APIs
# ---------------------------------------------------------------------

# 1. One Win applied to EVERY trace.
#
# Pick an A window. Only the two A traces should therefore contain the
# event response; B/C traces should mostly contain noise.
single_win = wins.get(0)

cut_single = traces.extract_win(
    single_win,
    align='ref',
)


# 2. Explicit relation.
#
# Upsample to 200 Hz while aligning every extracted piece to its window ref.
cut_matched = traces.extract_matched(
    wins,
    matches,
    align='ref',
    hz=200.0,
)


# 3. Cartesian product.
#
# Every trace is paired with every window, independently of metadata.
# 6 traces × 12 windows = 72 output traces.
cut_all = traces.extract_all(
    wins,
    align='ref',
    hz=50.0,
)


# 4. Convenience metadata matching.
#
# This should encode the same relation as Matches.from_meta(..., by='experiment').
cut_by = traces.extract_by(
    wins,
    by='experiment',
    align='ref',
    hz=100.0,
)


print('Output sizes')
print('------------')
print('extract_win:    ', len(cut_single))
print('extract_matched:', len(cut_matched))
print('extract_all:    ', len(cut_all))
print('extract_by:     ', len(cut_by))

assert len(cut_single) == 6
assert len(cut_matched) == 24
assert len(cut_all) == 72
assert len(cut_by) == 24


print('\nSampling')
print('--------')
print('extract_win:    ', cut_single.hz)
print('extract_matched:', cut_matched.hz)
print('extract_all:    ', cut_all.hz)
print('extract_by:     ', cut_by.hz)


print('\nMetadata from extract_by')
print('------------------------')
display(cut_by.meta)


# ---------------------------------------------------------------------
# Plot helpers
# ---------------------------------------------------------------------


def triggered_average(extracted):
    return np.nanmean(extracted.values, axis=0)


def plot_triggered_average(
    ax,
    extracted,
    *,
    title,
    show_individual=True,
):
    if show_individual:
        ax.plot(
            extracted.time,
            extracted.values.T,
            linewidth=0.6,
            alpha=0.12,
        )

    ax.plot(
        extracted.time,
        triggered_average(extracted),
        linewidth=2.5,
    )

    ax.axvline(
        0.0,
        linestyle='--',
        linewidth=1.0,
    )

    ax.set_title(f'{title}\nn={len(extracted)}, hz={extracted.hz:g}')
    ax.set_xlabel('aligned time [ms]')
    ax.set_ylabel('signal')


# ---------------------------------------------------------------------
# Triggered averages
# ---------------------------------------------------------------------

fig, axs = plt.subplots(
    2,
    2,
    figsize=(11, 7),
    constrained_layout=True,
)

plot_triggered_average(
    axs[0, 0],
    cut_single,
    title='extract_win — one A window over all traces',
)

plot_triggered_average(
    axs[0, 1],
    cut_matched,
    title='extract_matched — experiment matches only',
)

plot_triggered_average(
    axs[1, 0],
    cut_all,
    title='extract_all — Cartesian product',
)

plot_triggered_average(
    axs[1, 1],
    cut_by,
    title="extract_by(by='experiment')",
)

plt.show()


# ---------------------------------------------------------------------
# Alignment semantics
# ---------------------------------------------------------------------
#
# The window is [-500, +800] relative to ref.
# Therefore its midpoint is +150 ms relative to ref.
#
# So aligning to the midpoint instead of ref should shift the response
# LEFT by exactly 150 ms.

cut_ref = traces.extract_by(
    wins,
    by='experiment',
    align='ref',
    hz=100.0,
)

cut_mid = traces.extract_by(
    wins,
    by='experiment',
    align='mid',
    hz=100.0,
)

fig, ax = plt.subplots(
    figsize=(7, 4),
    constrained_layout=True,
)

ax.plot(
    cut_ref.time,
    triggered_average(cut_ref),
    label="align='ref'",
    linewidth=2,
)

ax.plot(
    cut_mid.time,
    triggered_average(cut_mid),
    label="align='mid'",
    linewidth=2,
)

ax.axvline(0.0, linestyle='--', linewidth=1)

ax.set(
    xlabel='aligned time [ms]',
    ylabel='triggered average',
    title='Alignment check — midpoint is 150 ms after ref',
)

ax.legend()
plt.show()


# ---------------------------------------------------------------------
# Numeric align is another useful smoke test.
#
# align=0.5 means exactly the same thing as align='mid'.
# ---------------------------------------------------------------------

cut_half = traces.extract_by(
    wins,
    by='experiment',
    align=0.5,
    hz=100.0,
)

np.testing.assert_allclose(
    cut_half.values,
    cut_mid.values,
    equal_nan=True,
)

np.testing.assert_allclose(
    cut_half.time,
    cut_mid.time,
)

print("align=0.5 and align='mid' agree.")

In [ ]:
fig, ax = plt.subplots(
    figsize=(7, 4),
    constrained_layout=True,
)


for k, group in cut_matched.groupby('experiment').items():
    ax.plot(
        group.time, group.values.T, linewidth=0.6, alpha=0.5, color=f'C{hash(k) % 10}'
    )
    ax.plot(
        group.time,
        triggered_average(group),
        label=f"align='ref', experiment={k}",
        linewidth=2,
        color=f'C{hash(k) % 10}',
        zorder=10,
    )

ax.set(
    xlabel='aligned time [ms]',
    ylabel='triggered average',
    title='Alignment check — midpoint is 150 ms after ref',
)

ax.legend()
plt.show()